# Water Asset Agent Prototype Review

这个 notebook 是一个独立测试原型入口：它会抓取 Google Earth Engine `water` 标签页下的全部数据集，构建结构化资产目录，然后根据用户问题自动检索候选资产、选择图层、生成 tile URL，并将图层叠加到底图上进行人工审核。

In [ ]:
%pip install -r ./requirements.txt

In [1]:
from pathlib import Path
import importlib
import sys
import pandas as pd
from IPython.display import display, Markdown

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import experiments.water_asset_agent.agent as water_asset_agent_module
importlib.reload(water_asset_agent_module)
from experiments.water_asset_agent.agent import WaterAssetAgentSystem

d:\Conda\envs\floodagent\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
import os
from dotenv import load_dotenv

ENV_PATH = PROJECT_ROOT / '.env'
load_dotenv(dotenv_path=ENV_PATH, override=True)

effective_project = os.getenv('GEE_PROJECT_ID') or os.getenv('PROJECT_ID')
effective_credentials = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')

display(Markdown(
    "## Runtime Env Check\n"
    f"- .env_path: `{ENV_PATH}`\n"
    f"- GEE_PROJECT_ID: `{os.getenv('GEE_PROJECT_ID')}`\n"
    f"- PROJECT_ID: `{os.getenv('PROJECT_ID')}`\n"
    f"- effective_project: `{effective_project}`\n"
    f"- GOOGLE_APPLICATION_CREDENTIALS: `{os.getenv('GOOGLE_APPLICATION_CREDENTIALS')}`\n"
    f"- effective_credentials: `{effective_credentials}`\n"
))

## Runtime Env Check
- .env_path: `c:\Users\admin\.codex\worktrees\9791\SatGPT-app\.env`
- GEE_PROJECT_ID: `flood-agent`
- PROJECT_ID: `flood-agent`
- effective_project: `flood-agent`
- GOOGLE_APPLICATION_CREDENTIALS: `D:\2025\.private-key.json`
- effective_credentials: `D:\2025\.private-key.json`


In [3]:
system = WaterAssetAgentSystem(project_root=PROJECT_ROOT)
assets = system.build_catalog(force_refresh=False)
display(Markdown(f"## Catalog Ready\nasset_count = `{len(assets)}`"))
catalog_df = pd.DataFrame([
    {
        'slug': item.slug,
        'title': item.title,
        'asset_id': item.asset_id,
        'asset_type': item.asset_type,
        'themes': ', '.join(item.themes),
        'spatial_scope': item.spatial_scope,
        'temporal_type': item.temporal_type,
    }
    for item in assets
])
catalog_df.head(20)

## Catalog Ready
asset_count = `95`

,slug,title,asset_id,asset_type,themes,spatial_scope,temporal_type
0,NOAA_CFSR,CFSR: Climate Forecast System Reanalysis | Ear...,NOAA/CFSR,ImageCollection,"surface_water, ocean, soil_water",global,time_series
1,NASA_ORNL_DAYMET_V4,Daymet V4: Daily Surface Weather and Climatolo...,NASA/ORNL/DAYMET_V4,ImageCollection,surface_water,global,daily
2,OPERA_DSWX_L3_V1_HLS,Dynamic Surface Water Extent from Harmonized L...,OPERA/DSWX/L3_V1/HLS,ImageCollection,surface_water,global,time_series
3,OPERA_DSWX_L3_V1_S1,Dynamic Surface Water Extent from Sentinel-1 (...,OPERA/DSWX/L3_V1/S1,ImageCollection,surface_water,global,time_series
4,GLCF_GLS_WATER,GLCF: Landsat Global Inland Water | Earth Engi...,GLCF/GLS_WATER,ImageCollection,surface_water,global,time_series
5,GLOBAL_FLOOD_DB_MODIS_EVENTS_V1,Global Flood Database v1 (2000-2018) | Earth E...,GLOBAL_FLOOD_DB/MODIS_EVENTS/V1,ImageCollection,"flood, surface_water",global,time_series
6,USGS_WBD_2017_HUC02,HUC02: USGS Watershed Boundary Dataset of Regi...,USGS/WBD/2017/HUC02,FeatureCollection,"surface_water, watershed",regional,static
7,USGS_WBD_2017_HUC04,HUC04: USGS Watershed Boundary Dataset of Subr...,USGS/WBD/2017/HUC04,FeatureCollection,"surface_water, watershed",regional,static
8,USGS_WBD_2017_HUC06,HUC06: USGS Watershed Boundary Dataset of Basi...,USGS/WBD/2017/HUC06,FeatureCollection,"surface_water, watershed",regional,static
9,USGS_WBD_2017_HUC08,HUC08: USGS Watershed Boundary Dataset of Subb...,USGS/WBD/2017/HUC08,FeatureCollection,"surface_water, watershed",regional,static


In [4]:
question = "请帮我看看 2024 年洞庭湖周边最适合叠加查看的 water 数据图层，并直接上图。"
result = system.ask(question=question, force_refresh_catalog=False)
review_map = system.last_map

display(Markdown(f"## Question\n{question}"))
display(Markdown(f"## Structured Query\n- need_visualization: `{result.structured_query.need_visualization}`\n- intent: `{result.structured_query.intent}`\n- themes: `{', '.join(result.structured_query.themes)}`\n- location_hint: `{result.structured_query.location_hint}`\n- start_date: `{result.structured_query.start_date}`\n- end_date: `{result.structured_query.end_date}`\n- used_llm: `{result.structured_query.used_llm}`\n\n{result.structured_query.answer}"))
display(Markdown(f"## GEE Auth\n- mode: `{result.gee_auth.get('mode')}`\n- project_id: `{result.gee_auth.get('project_id')}`\n- project_id_source: `{result.gee_auth.get('project_id_source')}`\n- account: `{result.gee_auth.get('account')}`\n- credentials_path: `{result.gee_auth.get('credentials_path')}`\n- tile_proxy_base_url: `{result.gee_auth.get('tile_proxy_base_url')}`"))
if result.map_error:
    display(Markdown(f"## Map Error\n{result.map_error}"))
    if 'serviceusage.services.use' in result.map_error:
        display(Markdown(
            "## Permission Fix Hint\n"
            "当前调用身份对目标项目缺少 `serviceusage.services.use`。\n\n"
            "建议优先处理：\n"
            "1. 在仓库根目录 `.env` 明确设置 `GEE_PROJECT_ID` 为你有权限的项目。\n"
            "2. 确保 `GOOGLE_APPLICATION_CREDENTIALS` 指向该项目下可用的 service account JSON。\n"
            "3. 若必须使用当前项目，请让项目 Owner 给调用账号授予 `roles/serviceusage.serviceUsageConsumer`。"
        ))

## Question
请帮我看看 2024 年洞庭湖周边最适合叠加查看的 water 数据图层，并直接上图。

## Structured Query
- need_visualization: `True`
- intent: `visualize`
- themes: `surface_water`
- location_hint: `洞庭湖`
- start_date: `2024-01-01`
- end_date: `2024-12-31`
- used_llm: `True`

请查看洞庭湖周边的水数据图层。

## GEE Auth
- mode: `service_account`
- project_id: `flood-agent`
- project_id_source: `env:GEE_PROJECT_ID`
- account: `flood-agent@flood-agent.iam.gserviceaccount.com`
- credentials_path: `D:\2025\.private-key.json`
- tile_proxy_base_url: `http://127.0.0.1:4418`

In [5]:
candidates_df = pd.DataFrame([
    {
        'slug': item.slug,
        'title': item.title,
        'asset_id': item.asset_id,
        'asset_type': item.asset_type,
        'themes': ', '.join(item.themes),
        'temporal_type': item.temporal_type,
        'priority': item.priority,
    }
    for item in result.candidates
])
display(Markdown("## Candidate Assets"))
candidates_df

## Candidate Assets

,slug,title,asset_id,asset_type,themes,temporal_type,priority
0,HYCOM_sea_temp_salinity,"HYCOM: Hybrid Coordinate Ocean Model, Water Te...",HYCOM/sea_temp_salinity,ImageCollection,"surface_water, ocean",time_series,1
1,JRC_GSW1_4_GlobalSurfaceWater,"JRC Global Surface Water Mapping Layers, v1.4 ...",JRC/GSW1_4/GlobalSurfaceWater,Image,"surface_water, ocean",static,1
2,JRC_GSW1_4_MonthlyRecurrence,"JRC Monthly Water Recurrence, v1.4 | Earth Eng...",JRC/GSW1_4/MonthlyRecurrence,ImageCollection,surface_water,monthly,1
3,NOAA_CFSR,CFSR: Climate Forecast System Reanalysis | Ear...,NOAA/CFSR,ImageCollection,"surface_water, ocean, soil_water",time_series,1
4,OPERA_DSWX_L3_V1_HLS,Dynamic Surface Water Extent from Harmonized L...,OPERA/DSWX/L3_V1/HLS,ImageCollection,surface_water,time_series,1
5,OPERA_DSWX_L3_V1_S1,Dynamic Surface Water Extent from Sentinel-1 (...,OPERA/DSWX/L3_V1/S1,ImageCollection,surface_water,time_series,1
6,GLOBAL_FLOOD_DB_MODIS_EVENTS_V1,Global Flood Database v1 (2000-2018) | Earth E...,GLOBAL_FLOOD_DB/MODIS_EVENTS/V1,ImageCollection,"flood, surface_water",time_series,1
7,JRC_GSW1_4_Metadata,"JRC Global Surface Water Metadata, v1.4 | Eart...",JRC/GSW1_4/Metadata,Image,surface_water,static,1
8,JRC_GSW1_4_MonthlyHistory,"JRC Monthly Water History, v1.4 | Earth Engine...",JRC/GSW1_4/MonthlyHistory,ImageCollection,surface_water,monthly,1
9,JRC_GSW1_4_YearlyHistory,"JRC Yearly Water Classification History, v1.4 ...",JRC/GSW1_4/YearlyHistory,ImageCollection,surface_water,yearly,1


In [6]:
selected_df = pd.DataFrame([
    {
        'slug': item.slug,
        'title': item.title,
        'asset_id': item.asset_id,
        'asset_type': item.asset_type,
        'themes': ', '.join(item.themes),
        'official_url': item.official_url,
    }
    for item in result.selected_assets
])
display(Markdown("## Selected Assets"))
selected_df

## Selected Assets

,slug,title,asset_id,asset_type,themes,official_url
0,HYCOM_sea_temp_salinity,"HYCOM: Hybrid Coordinate Ocean Model, Water Te...",HYCOM/sea_temp_salinity,ImageCollection,"surface_water, ocean",https://developers.google.com/earth-engine/dat...
1,JRC_GSW1_4_GlobalSurfaceWater,"JRC Global Surface Water Mapping Layers, v1.4 ...",JRC/GSW1_4/GlobalSurfaceWater,Image,"surface_water, ocean",https://developers.google.com/earth-engine/dat...
2,JRC_GSW1_4_MonthlyRecurrence,"JRC Monthly Water Recurrence, v1.4 | Earth Eng...",JRC/GSW1_4/MonthlyRecurrence,ImageCollection,surface_water,https://developers.google.com/earth-engine/dat...


In [7]:
layers_df = pd.DataFrame(result.rendered_layers)
display(Markdown("## Rendered Layers"))
layers_df

## Rendered Layers

,layer_id,asset_id,asset_type,title,browser_tile_url,proxy_tile_url,earth_engine_tile_url,sample_browser_tile_url,sample_proxy_tile_url,sample_earth_engine_tile_url,...,bounds,vis_params_used,vis_recipe_source,validator_notes,collection_strategy,collection_filter_notes,available_bands,official_example_vis,tile_loading_mode,official_url
0,hycom_sea_temp_salinity,HYCOM/sea_temp_salinity,ImageCollection,"HYCOM: Hybrid Coordinate Ocean Model, Water Te...",https://earthengine.googleapis.com/v1/projects...,http://127.0.0.1:4418/ee-tiles/hycom_sea_temp_...,https://earthengine.googleapis.com/v1/projects...,https://earthengine.googleapis.com/v1/projects...,http://127.0.0.1:4418/ee-tiles/hycom_sea_temp_...,https://earthengine.googleapis.com/v1/projects...,...,"{'south': 28.6264881, 'north': 29.5152175, 'we...","{'palette': ['000000', '005aff', '43c8c8', 'ff...",official_example,[palette_requires_single_band_auto_selected],region_and_date,"[filter_bounds_applied, filter_date_between_ap...","[water_temp_0, salinity_0, water_temp_2, salin...","{'palette': ['000000', '005aff', '43c8c8', 'ff...",direct_earth_engine_tiles,https://developers.google.com/earth-engine/dat...
1,jrc_gsw1_4_globalsurfacewater,JRC/GSW1_4/GlobalSurfaceWater,Image,"JRC Global Surface Water Mapping Layers, v1.4 ...",https://earthengine.googleapis.com/v1/projects...,http://127.0.0.1:4418/ee-tiles/jrc_gsw1_4_glob...,https://earthengine.googleapis.com/v1/projects...,https://earthengine.googleapis.com/v1/projects...,http://127.0.0.1:4418/ee-tiles/jrc_gsw1_4_glob...,https://earthengine.googleapis.com/v1/projects...,...,"{'south': 28.6264881, 'north': 29.5152175, 'we...","{'bands': ['occurrence'], 'palette': ['ffffff'...",official_example,[],single_image,[],"[occurrence, change_abs, change_norm, seasonal...","{'bands': ['occurrence'], 'palette': ['ffffff'...",direct_earth_engine_tiles,https://developers.google.com/earth-engine/dat...
2,jrc_gsw1_4_monthlyrecurrence,JRC/GSW1_4/MonthlyRecurrence,ImageCollection,"JRC Monthly Water Recurrence, v1.4 | Earth Eng...",https://earthengine.googleapis.com/v1/projects...,http://127.0.0.1:4418/ee-tiles/jrc_gsw1_4_mont...,https://earthengine.googleapis.com/v1/projects...,https://earthengine.googleapis.com/v1/projects...,http://127.0.0.1:4418/ee-tiles/jrc_gsw1_4_mont...,https://earthengine.googleapis.com/v1/projects...,...,"{'south': 28.6264881, 'north': 29.5152175, 'we...","{'bands': ['monthly_recurrence'], 'palette': [...",official_example,[],region_only,"[filter_bounds_applied, fallback_strategy:regi...","[monthly_recurrence, has_observations]","{'bands': ['monthly_recurrence'], 'palette': [...",direct_earth_engine_tiles,https://developers.google.com/earth-engine/dat...


In [8]:
display(Markdown("## Raw Tile URLs"))
if layers_df.empty:
    print('No rendered layers.')
else:
    for idx, row in layers_df.iterrows():
        print(f"[{idx}] {row.get('title', '')}")
        print('browser_tile_url_template:')
        print(row.get('browser_tile_url', 'NO_BROWSER_TILE_URL'))
        print('earth_engine_tile_url_template:')
        print(row.get('earth_engine_tile_url', 'NO_EARTH_ENGINE_TILE_URL'))
        print('sample_browser_tile_url:')
        print(row.get('sample_browser_tile_url', 'NO_SAMPLE_BROWSER_TILE_URL'))
        print('sample_earth_engine_tile_url:')
        print(row.get('sample_earth_engine_tile_url', 'NO_SAMPLE_EARTH_ENGINE_TILE_URL'))
        print('-' * 120)

## Raw Tile URLs

[0] HYCOM: Hybrid Coordinate Ocean Model, Water Temperature and Salinity | Earth Engine Data Catalog | Google for Developers
browser_tile_url_template:
https://earthengine.googleapis.com/v1/projects/flood-agent/maps/15396ac0154a41174c6178eb10115d0c-b6996c9676aeabacf0c0c618e0a4b6c7/tiles/{z}/{x}/{y}
earth_engine_tile_url_template:
https://earthengine.googleapis.com/v1/projects/flood-agent/maps/15396ac0154a41174c6178eb10115d0c-b6996c9676aeabacf0c0c618e0a4b6c7/tiles/{z}/{x}/{y}
sample_browser_tile_url:
https://earthengine.googleapis.com/v1/projects/flood-agent/maps/15396ac0154a41174c6178eb10115d0c-b6996c9676aeabacf0c0c618e0a4b6c7/tiles/1/1/0
sample_earth_engine_tile_url:
https://earthengine.googleapis.com/v1/projects/flood-agent/maps/15396ac0154a41174c6178eb10115d0c-b6996c9676aeabacf0c0c618e0a4b6c7/tiles/1/1/0
------------------------------------------------------------------------------------------------------------------------
[1] JRC Global Surface Water Mapping Layers, v1.4 | Earth En

In [9]:
display(Markdown("## Sample Tile Request Test"))
if layers_df.empty:
    print('No layers to test.')
else:
    import requests
    test_url = layers_df.iloc[0].get('sample_browser_tile_url')
    print('GET', test_url)
    resp = requests.get(test_url, timeout=30)
    print('status =', resp.status_code)
    print('content_type =', resp.headers.get('Content-Type'))
    print('bytes =', len(resp.content))

## Sample Tile Request Test

GET https://earthengine.googleapis.com/v1/projects/flood-agent/maps/15396ac0154a41174c6178eb10115d0c-b6996c9676aeabacf0c0c618e0a4b6c7/tiles/1/1/0
status = 200
content_type = image/png
bytes = 334


In [10]:
if review_map is not None:
    display(Markdown("## Map Preview"))
    display(review_map)
else:
    display(Markdown("## Map Preview\nNo map generated."))

## Map Preview